In [2]:
import json
from pathlib import Path

def load_jade_file(path: Path) -> tuple[dict, list]:
    """Return (db, order) from a JADE export JSON file."""
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    return raw["db"], raw["order"]
 
 
# ── Load each file into its own named objects ─────────────────────────────────
cases_first, order_first = load_jade_file("input/FIRST 399 cases jade-cases-2026-05-13.json")
cases_last,  order_last  = load_jade_file("input/Last 387 cases jade-cases-2026-05-13.json")
cases_extra, order_extra = load_jade_file("input/missing cases.json")

# ── Combined view (optional convenience) ─────────────────────────────────────
cases_all = {**cases_first, **cases_last, **cases_extra}   # Last_387 wins on any citation clash
order_all = order_first + order_last + order_extra

In [3]:
print(cases_all)

{'[1998] HCA 28': {'citation': '[1998] HCA 28', 'catchwords': 'Statutes - Construction - Reconciliation of conflicting provisions - Intention of legislature - Presumption that provisions intended to achieve consistent goals - Leading and subordinate provisions - Grammatical meaning and legal meaning. Statutes - Construction - Acts done in breach of a condition regulating a statutory power - Whether invalid - Mandatory and directory provisions - Purpose-based test. Media law - Television - Regulation of programming - Australian Broadcasting Authority - Standard prescribing Australian content requirements - Whether inconsistent with legislative requirement that functions be performed consistently with Australia\'s international obligations. Media law - Television - Regulation of programming - Australian Broadcasting Authority - Power to make standards that "relate to ... the Australian content of programs" - Whether restricted to standards conferring preferential treatment. Trade law - A

In [4]:
print(f"cases_first : {len(cases_first):>4} cases  |  order_first : {len(order_first):>4} entries")
print(f"cases_last  : {len(cases_last):>4} cases  |  order_last  : {len(order_last):>4} entries")
print(f"cases_all   : {len(cases_all):>4} cases  |  order_all   : {len(order_all):>4} entries")

# Show the keys available on every case record
sample_key = order_first[0]
sample = cases_first[sample_key]
print(f"\nSample case  : {sample_key}")
print(f"Fields       : {', '.join(sample.keys())}")
print(f"Citation     : {sample['citation']}")
print(f"Bench        : {sample.get('bench', '(none)')}")

cases_first :  395 cases  |  order_first :  395 entries
cases_last  :  384 cases  |  order_last  :  384 entries
cases_all   :  781 cases  |  order_all   :  781 entries

Sample case  : [1998] HCA 28
Fields       : citation, catchwords, bench, judgmentAuthors, paragraphs, autoNotes, userNotes, importedAt
Citation     : [1998] HCA 28
Bench        : ['BRENNAN', 'McHUGH', 'GUMMOW', 'KIRBY', 'HAYNE']


In [5]:
import pandas as pd
from pathlib import Path
 
FILE = Path("input/original list .xlsx")
 
# ── Load ──────────────────────────────────────────────────────────────────────
_raw = pd.read_excel(FILE, sheet_name="Sheet1")
 
# Drop all-empty trailing columns (Unnamed: 13 … Unnamed: 26)
_raw = _raw.dropna(axis=1, how="all")
_raw = _raw.loc[:, ~_raw.columns.str.startswith("Unnamed:")]
 
# ── Rename to snake_case ──────────────────────────────────────────────────────
_col_map = {
    "Type":                     "type",
    "KeyCite Treatment":        "keycite_treatment",
    "KeyCite URL":              "keycite_url",
    "Citing References Count":  "citing_references_count",
    "Citing References URL":    "citing_references_url",
    "Title":                    "title",
    "Document URL":             "document_url",
    "Court Line":               "court",
    "Jurisdiction":             "jurisdiction",
    "Filed Date":               "filed_date",
    "Citation":                 "citation",
    "Parallel Cite":            "parallel_cite",
    "Summary":                  "summary",
}
df = _raw.rename(columns=_col_map)
 
# ── Type coercions ────────────────────────────────────────────────────────────
# Count column: float → nullable integer
df["citing_references_count"] = df["citing_references_count"].astype("Int64")
 
# Filed date: datetime64 → plain Python date
df["filed_date"] = pd.to_datetime(df["filed_date"]).dt.date
 


In [6]:
# ── Sanity check ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
    valid_dates = pd.to_datetime(df["filed_date"], errors="coerce").dropna()
    print(f"Date range: {valid_dates.min().date()} → {valid_dates.max().date()}")
    print(f"Citations range: {df['citing_references_count'].min():,} – "
          f"{df['citing_references_count'].max():,}")
    print()
    print(df[["citation", "title", "filed_date", "citing_references_count"]].head(5).to_string(index=False))

Rows: 794  |  Columns: 13
Date range: 1997-09-25 → 2012-10-05
Citations range: 1 – 6,939

     citation                                                            title filed_date  citing_references_count
[1998] HCA 28         Project Blue Sky Inc v Australian Broadcasting Authority 1998-04-28                     6939
 [2003] HCA 2                               Plaintiff S157/2002 v Commonwealth 2003-02-04                     5163
[2003] HCA 22                                                      Fox v Percy 2003-04-30                     5053
[2001] HCA 30       Minister for Immigration and Multicultural Affairs v Yusuf 2001-05-31                     4344
[2009] HCA 27 Aon Risk Services Australia Ltd v Australian National University 2009-08-05                     4319


In [7]:
"""#testing stuff
print(cases_all[order_all[1]])
print(order_all[:5])"""


'#testing stuff\nprint(cases_all[order_all[1]])\nprint(order_all[:5])'

We now need to do some QC stuff - checking what is missing

In [9]:

intersection = set(order_all) & set(df['citation'])
print(f"Number of cases in both datasets: {len(intersection)}")
# print(test.keys())
overlapping_cases = df['citation'][~df['citation'].isin(intersection)]
missing_cases = set(order_all) - set(df['citation'])
print(f"Cases in order_all but not in df: {len(missing_cases)}")
print(missing_cases)
print(f"Cases in df but not in order_all: {len(overlapping_cases)}")
print(overlapping_cases)

missing_cases = list(missing_cases)
missing_indices = [order_all.index(case) for case in missing_cases]
print(missing_indices)
print([order_all[i] for i in missing_indices])

print(cases_all[order_all[missing_indices[0]]])

Number of cases in both datasets: 780
Cases in order_all but not in df: 1
{'[1998] HCA 25'}
Cases in df but not in order_all: 14
84                 [1997] HCA 56
155                [1997] HCA 53
190                [1997] HCA 49
239                [1997] HCA 47
242                [1998] HCA 58
298                [1998] HCA 76
363                [1999] HCA 57
382                [1997] HCA 54
745                [1997] HCA 55
748    HC C13/1997, 3 April 1998
754                [1997] HCA 48
790     HC C13/1997, 20 May 1998
791        HC, 30 September 1997
793                          NaN
Name: citation, dtype: str
[737]
['[1998] HCA 25']
{'citation': '[1998] HCA 25', 'catchwords': 'Attorney-General (Cth) v Tse Chu-Fai Extradition – Application on behalf of extradition country for issue of warrant of arrest – Meaning of "extradition country" – Whether "Hong Kong" specified in Extradition (Hong Kong) Regulations a "territory for the international relations of which a country is responsible" 

In [10]:
intersection = set(cases_all.keys()) - set(order_all)
print(f"Cases in cases_all but not in order_all: {len(intersection)}")
print(intersection)

seen = set()
duplicates = [x for x in order_all if x in seen or seen.add(x)]
print(f"Duplicate citations in order_all: {duplicates}")

Cases in cases_all but not in order_all: 0
set()
Duplicate citations in order_all: []


In [ ]:
# rename keys 
"""def rename_case(old_citation, new_citation, cases_all, order_all):
    if old_citation in cases_all:
        old_index = order_all.index(old_citation)
        order_all[old_index] = None
        order_all.insert(old_index, new_citation)
        cases_all[new_citation] = cases_all.pop(old_citation)
    cases_all[new_citation]['citation'] = new_citation
    return cases_all, order_all

cases_all, order_all = rename_case('[2010] HCA 0',"[2010] HCA 35", cases_all, order_all)"""


updated intersection test

In [ ]:

"""intersection = set(order_all) & set(df['citation'])
print(f"Number of cases in both datasets: {len(intersection)}")
# print(test.keys())
overlapping_cases = df['citation'][~df['citation'].isin(intersection)]
missing_cases = set(order_all) - set(df['citation'])
print(f"Cases in order_all but not in df: {len(missing_cases)}")
print(missing_cases)
print(f"Cases in df but not in order_all: {len(overlapping_cases)}")
print(overlapping_cases)"""

In [ ]:
"""import re
from collections import defaultdict

# ── Tagging approach ──────────────────────────────────────────────────────────
# HCA catchwords follow "Topic – subtopic – ...". The approach:
#   PRIMARY:   patterns matched against the leading topic segment (high precision)
#   SECONDARY: two keywords must co-occur anywhere in the full catchwords text
# A case is tagged if it satisfies PRIMARY *or* any SECONDARY pair.
# ─────────────────────────────────────────────────────────────────────────────

def _r(pat):
    return re.compile(pat, re.IGNORECASE)

# Recognised top-level area names that may appear appended to a case name
# in the first catchword segment (e.g. "Smith v Jones Criminal law – ...")
_AREA_IN_FIRST_SEG = re.compile(
    r'(Criminal law|Constitutional law|Administrative law|Negligence|Torts?|'
    r'Statutes?|Contracts?|Trade practices|Evidence|Equity|Trusts?|Real property|'
    r'Corporations?|Taxation|Immigration|Refugees?|Family law|Defamation|'
    r'Patents?|Copyright|Native title|Bankruptcy|Insolvency|Sentencing|'
    r'Industrial law|Workers.? compensation|Discrimination|Aviation|Shipping|'
    r'Mining|Planning|Extradition|Practice and procedure|Damages|Stamp duty|'
    r'Customs|Land tax|Limitation of actions)',
    re.IGNORECASE
)

def leading_topic(cw: str) -> str:
    """
    Extract the first recognisable legal topic from catchwords.
    Handles the common HCA pattern where the case name prefixes the first
    topic segment: "Smith v Jones Criminal law – subtopic – ..."
    """
    parts = re.split(r' [–\-] ', cw)

    # First try: find a segment that looks like a clean topic (no case name)
    for p in parts:
        p = p.strip()
        if re.match(r'^[A-Z][a-z]', p) and ' v ' not in p and len(p) < 100:
            return p

    # Fallback: first segment may be "CaseName Area" — extract the area name
    if parts:
        m = _AREA_IN_FIRST_SEG.search(parts[0])
        if m:
            return m.group(0)

    return parts[0] if parts else ''


LAW_AREAS = {

    "Constitutional Law": dict(
        primary=[
            _r(r"constitutional law"),
            _r(r"legislative power"),
            _r(r"judicial power of (the )?commonwealth"),
            _r(r"inconsistency between commonwealth"),
            _r(r"chapter iii|ch iii"),
            _r(r"separation of powers"),
            _r(r"compulsory acquisition"),
            _r(r"acquisition of property"),
            _r(r"manner and form"),
        ],
        secondary=[
            (_r(r"\bconstitution\b"),          _r(r"\bs\s?5[19]\b|\bs\s?7[56]\b|\bs\s?109\b|commonwealth parliament|state parliament|chapter iii")),
            (_r(r"federal jurisdiction"),      _r(r"vesting|chapter iii|ch iii|\bconstitution\b")),
            (_r(r"just terms"),                _r(r"acquisition|property")),
            (_r(r"\bjudicial power\b"),        _r(r"commonwealth|chapter iii|ch iii")),
            (_r(r"\bparliament\b"),            _r(r"powers|manner and form|elections?|state parliament|commonwealth parliament")),
            (_r(r"constitutional"),            _r(r"validity|invalid|power|inconsistency")),
            (_r(r"trial by jury"),             _r(r"constitutional|\bconstitution\b|chapter iii|indictable")),
        ],
    ),

    "Administrative Law": dict(
        primary=[
            _r(r"administrative law"),
            _r(r"judicial review"),
            _r(r"jurisdictional error"),
            _r(r"procedural fairness"),
            _r(r"natural justice"),
            _r(r"prerogative writ"),
            _r(r"privative clause"),
            _r(r"freedom of information"),
        ],
        secondary=[
            (_r(r"\bbias\b"),                  _r(r"court|judge|tribunal|judicial|disqualif|apprehension")),
            (_r(r"\bstanding\b"),              _r(r"judicial review|administrative|injunctive relief|ultra vires")),
            (_r(r"ultra vires"),               _r(r"statutory power|administrative|tribunal")),
            (_r(r"delegated legislation"),     _r(r"validity|ultra vires|statutory power")),
            (_r(r"\bcertiorari\b|\bmandamus\b|\bprohibition\b"), _r(r"review|tribunal|decision")),
            (_r(r"courts and judges"),         _r(r"bias|disqualif|apprehension")),
            (_r(r"apprehension of bias"),      _r(r"court|judge|judicial|disqualif")),
        ],
    ),

    "Criminal Law": dict(
        primary=[
            _r(r"criminal law"),
            _r(r"criminal procedure"),
            _r(r"criminal law and procedure"),
            _r(r"\bsentencing\b"),
            _r(r"\bmurder\b"),
            _r(r"\bmanslaughter\b"),
            _r(r"sexual offence"),
            _r(r"double jeopardy"),
            _r(r"military law"),
            _r(r"appeal against conviction"),
            _r(r"appeal against sentence"),
            _r(r"\bextradition\b"),
            _r(r"\bdefamation\b"),   # defamation is a criminal law topic in some HCA cases
        ],
        secondary=[
            (_r(r"\bsentence\b|\bsentencing\b"), _r(r"criminal|offend|convicted|guilty|plea|jury|imprisonment|parole|non.parole|proviso|quash")),
            (_r(r"\bconviction\b"),          _r(r"criminal|offend|jury|verdict|guilty|appeal|quash")),
            (_r(r"directions? to jury|trial by jury"), _r(r"criminal|offend|verdict|misdirection")),
            (_r(r"\bproviso\b"),             _r(r"criminal|conviction|verdict|jury|appeal|miscarriage")),
            (_r(r"\bindictment\b|\bplea\b"), _r(r"criminal|offend|guilty|jury")),
            (_r(r"\bconspiracy\b"),          _r(r"criminal|offend|defraud|possess")),
            (_r(r"\bdefences?\b"),           _r(r"criminal|offend|guilty|charge")),
            (_r(r"the queen|the crown"),     _r(r"criminal|offend|sentence|conviction|verdict|jury|guilty")),
            (_r(r"onus of proof"),           _r(r"criminal|offend|guilty|charge")),
            (_r(r"\bextradition\b"),         _r(r"criminal|offend|warrant|surrender|eligible")),
        ],
    ),

    "Immigration & Refugees": dict(
        primary=[
            _r(r"\brefugees?\b"),
            _r(r"\bimmigration\b"),
            _r(r"protection visa"),
            _r(r"refugee review tribunal"),
            _r(r"citizenship and migration"),
            _r(r"unlawful non.citizen"),
        ],
        secondary=[
            (_r(r"protection visa"),          _r(r"migration act|refugee|minister for immigration")),
            (_r(r"well.founded fear"),        _r(r"refugee|persecution|protection visa")),
            (_r(r"\bpersecution\b"),          _r(r"refugee|protection visa|migration act")),
            (_r(r"minister for immigration"), _r(r"visa|refugee|migration act|detention")),
            (_r(r"\bdetention\b"),            _r(r"immigration|migration act|unlawful non.citizen")),
            (_r(r"\bcitizenship\b"),          _r(r"migration act|immigration|naturalisation")),
            (_r(r"migration act"),            _r(r"visa|refugee|detention|deportation|immigration")),
        ],
    ),

    "Torts": dict(
        primary=[
            _r(r"\bnegligence\b"),
            _r(r"\btorts?\b"),
            _r(r"duty of care"),
            _r(r"malicious prosecution"),
            _r(r"medical negligence"),
            _r(r"\bdefamation\b"),
        ],
        secondary=[
            (_r(r"breach of duty"),           _r(r"negligence|duty of care|tort")),
            (_r(r"\bcausation\b"),            _r(r"negligence|duty of care|tort|personal injur")),
            (_r(r"contributory negligence"),  _r(r"negligence|duty of care|tort")),
            (_r(r"vicarious liability"),      _r(r"negligence|tort|employer|employee")),
            (_r(r"personal injuries"),        _r(r"negligence|duty of care|tort")),
            (_r(r"nervous shock|psychiatric injury"), _r(r"negligence|duty of care|tort")),
            (_r(r"\bdamages\b"),              _r(r"negligence|duty of care|defamation|tort|wrongful death|personal injur")),
            (_r(r"\bnuisance\b|\btrespass\b|\bmisfeasance\b"), _r(r"tort|negligence|duty")),
            (_r(r"standard of care"),         _r(r"negligence|duty of care|tort")),
            (_r(r"wrongful death"),           _r(r"damages|negligence|tort")),
            (_r(r"qualified privilege"),      _r(r"defamation|publication|imputation")),
        ],
    ),

    "Contract & Commercial Law": dict(
        primary=[
            _r(r"\bcontract\b"),
            _r(r"trade practices"),
            _r(r"misleading or deceptive conduct"),
            _r(r"sale of goods"),
            _r(r"banker and customer"),
            _r(r"unjust enrichment"),
            _r(r"\brestitution\b"),
            _r(r"restrictive trade practices"),
        ],
        secondary=[
            (_r(r"\bestoppel\b"),             _r(r"contract|commercial|representation|promissory")),
            (_r(r"\bpenalty\b"),              _r(r"contract|breach|liquidated")),
            (_r(r"unconscionable"),           _r(r"contract|commercial|trade practices|equity")),
            (_r(r"\bindemnity\b"),            _r(r"contract|agreement|commercial|insurance")),
            (_r(r"\bguarantee\b"),            _r(r"contract|agreement|commercial|bank")),
            (_r(r"principal and agent|\bagency\b"), _r(r"contract|commercial|authority")),
            (_r(r"\binsurance\b"),            _r(r"contract|policy|indemnity|liability|premium")),
            (_r(r"\bconsumer\b"),             _r(r"trade practices|contract|misleading|credit")),
            (_r(r"\bdamages\b"),              _r(r"contract|breach|trade practices|misleading")),
            (_r(r"misleading"),               _r(r"trade practices|deceptive|conduct|consumer")),
            (_r(r"restraint of trade"),       _r(r"contract|trade practices|competition")),
            (_r(r"franchis"),                 _r(r"contract|commercial|agreement|code")),
            (_r(r"market power|market definition|exclusionary"), _r(r"trade practices|competition|monopol")),
            (_r(r"intention to create"),      _r(r"contract|contractual|relations")),
            (_r(r"\bloan\b"),                 _r(r"contract|agreement|commercial|bank|lend")),
            (_r(r"trade practices act"),      _r(r"misleading|deceptive|conduct|damages|contravention")),
        ],
    ),

    "Statutory Interpretation": dict(
        primary=[
            _r(r"statutory interpretation"),
            _r(r"statutory construction"),
            _r(r"acts of parliament"),
        ],
        secondary=[
            (_r(r"\bstatutes\b"),             _r(r"construction|interpretation|meaning|purpose|intention of (the )?legislat|parliament")),
            (_r(r"\binterpretation\b"),       _r(r"acts? of parliament|statutory|extrinsic|legislative purpose|plain meaning|intention of")),
            (_r(r"extrinsic materials"),      _r(r"interpretation|statutory|acts? of parliament")),
            (_r(r"legislative purpose"),      _r(r"interpretation|statutory|construction")),
            (_r(r"words and phrases"),        _r(r"meaning|definition|interpretation|construction|statutory")),
        ],
    ),

    "Evidence": dict(
        primary=[
            _r(r"\bevidence\b"),
            _r(r"\badmissibility\b"),
            _r(r"legal professional privilege"),
            _r(r"tendency evidence"),
            _r(r"propensity evidence"),
        ],
        secondary=[
            (_r(r"\bhearsay\b"),              _r(r"evidence|admissib")),
            (_r(r"\bcredibility\b"),          _r(r"evidence|witness|admissib")),
            (_r(r"\bcorroboration\b"),        _r(r"evidence|witness")),
            (_r(r"expert evidence|opinion evidence"), _r(r"admissib|expert")),
            (_r(r"burden of proof|standard of proof|onus of proof"), _r(r"evidence|trial")),
            (_r(r"\bprivilege\b"),            _r(r"evidence|legal professional|without prejudice|spousal")),
            (_r(r"without prejudice|spousal privilege"), _r(r"evidence|privilege")),
            (_r(r"\brelevance\b|\brelevant\b"), _r(r"evidence|admissib|probative")),
            (_r(r"prior inconsistent"),       _r(r"evidence|witness|statement")),
        ],
    ),

    "Property & Equity": dict(
        primary=[
            _r(r"real property"),
            _r(r"\btorrens\b"),
            _r(r"\bequity\b"),
            _r(r"\btrusts?\b"),
            _r(r"land tax"),
            _r(r"succession"),
            _r(r"\bmortgages?\b"),
        ],
        secondary=[
            (_r(r"\bproperty\b"),             _r(r"real property|torrens|indefeasib|title|conveyance|freehold|leasehold|land|vesting")),
            (_r(r"\blease\b"),               _r(r"real property|land|torrens|landlord|tenant|rent|term")),
            (_r(r"\beasement\b"),             _r(r"real property|land|title")),
            (_r(r"\bmortgage\b"),             _r(r"real property|land|torrens|security|charge|receiver")),
            (_r(r"indefeasibility"),          _r(r"torrens|real property|title")),
            (_r(r"\bfiduciary\b"),            _r(r"equity|trust|duty")),
            (_r(r"constructive trust|resulting trust"), _r(r"equity|trust")),
            (_r(r"specific performance"),     _r(r"equity|contract|land")),
            (_r(r"\binjunction\b"),           _r(r"equity|interlocutory|restraint")),
            (_r(r"family provision"),         _r(r"succession|will|estate|probate")),
            (_r(r"\bwill\b|\bestate\b"),      _r(r"succession|probate|executor|testator")),
            (_r(r"receivers? and managers?"), _r(r"mortgage|property|land|security")),
            (_r(r"vesting of property"),      _r(r"property|unincorpor|trust|statute")),
        ],
    ),

    "Corporations & Insolvency": dict(
        primary=[
            _r(r"\bcorporations?\b"),
            _r(r"corporations law"),
            _r(r"company law"),
            _r(r"winding up"),
            _r(r"\binsolvency\b"),
            _r(r"\bbankruptcy\b"),
        ],
        secondary=[
            (_r(r"\bdirectors?\b"),           _r(r"corporation|company|duties|fiduciary|breach")),
            (_r(r"\bshareholders?\b"),        _r(r"corporation|company|shares?")),
            (_r(r"\bliquidation\b"),          _r(r"corporation|company|winding up|insolvency")),
            (_r(r"\breceivership\b"),         _r(r"corporation|company|insolvency")),
            (_r(r"\bsecurities?\b"),          _r(r"corporation|company|asic|disclosure|prospectus")),
            (_r(r"\btakeover\b"),             _r(r"corporation|company|shares?")),
            (_r(r"\basic\b"),                 _r(r"corporation|company|disclosure|securities")),
            (_r(r"external administration"), _r(r"corporation|company|deed of company")),
            (_r(r"deed of company arrangement"), _r(r"corporation|company|administration")),
            (_r(r"\bcharge\b"),               _r(r"corporations?|company|insolvency|security|creditor")),
            (_r(r"litigation funding|champert"), _r(r"corporations?|company|insolvency")),
        ],
    ),

    "Taxation": dict(
        primary=[
            _r(r"\btaxation\b"),
            _r(r"income tax"),
            _r(r"stamp duty"),
            _r(r"taxes and duties"),
            _r(r"allowable deductions"),
            _r(r"assessable income"),
            _r(r"tax avoidance"),
            _r(r"fringe benefits tax"),
            _r(r"land tax"),
            _r(r"\bcustoms\b"),
        ],
        secondary=[
            (_r(r"commissioner of taxation"), _r(r"income tax|assessable|deductions|tax")),
            (_r(r"capital gains"),            _r(r"tax|assessable|income tax")),
            (_r(r"\bgst\b"),                  _r(r"tax|supply|goods and services")),
            (_r(r"\bexcise\b"),               _r(r"duty|tax|commonwealth|importation")),
            (_r(r"\bsuperannuation\b"),       _r(r"tax|fund|contribution")),
            (_r(r"valuation of land"),        _r(r"land tax|rating|municipal")),
            (_r(r"dutiable"),                 _r(r"stamp duty|duty|tax")),
            (_r(r"importation"),              _r(r"customs|duty|prohibited|excise")),
        ],
    ),

    "Family Law": dict(
        primary=[
            _r(r"family law"),
            _r(r"family court"),
            _r(r"child support"),
            _r(r"child abduction"),
        ],
        secondary=[
            (_r(r"\bcustody\b|\bguardianship\b"), _r(r"family|child|parenting|matrimonial")),
            (_r(r"\bparenting\b"),            _r(r"family|child|order|court")),
            (_r(r"property settlement"),      _r(r"family|matrimonial|marriage|de facto")),
            (_r(r"\bmaintenance\b"),          _r(r"family|matrimonial|marriage|spouse")),
            (_r(r"\bmarriage\b|\bdivorce\b"), _r(r"family|matrimonial|family law")),
            (_r(r"de facto"),                 _r(r"family|relationship|property")),
            (_r(r"habitually resident"),      _r(r"child|family|abduction|convention")),
        ],
    ),

    "Intellectual Property": dict(
        primary=[
            _r(r"intellectual property"),
            _r(r"\bcopyright\b"),
            _r(r"\bpatents?\b"),
            _r(r"trade marks?"),
            _r(r"\btrademark\b"),
            _r(r"passing off"),
        ],
        secondary=[
            (_r(r"\binfringement\b"),         _r(r"copyright|patent|trade mark|intellectual property")),
            (_r(r"inventive step|\bobviousness\b|\brevocation\b"), _r(r"patent")),
            (_r(r"\blicence\b|\blicense\b"),  _r(r"copyright|patent|trade mark|intellectual property")),
            (_r(r"phonographic performance"), _r(r"copyright|licence|broadcast")),
        ],
    ),

    "Native Title & Aboriginal Law": dict(
        primary=[
            _r(r"native title"),
            _r(r"\baboriginals?\b"),
            _r(r"land rights"),
        ],
        secondary=[
            (_r(r"\bindigenous\b"),           _r(r"native title|land rights|aboriginal|traditional")),
            (_r(r"pastoral lease"),           _r(r"native title|aboriginal|land")),
            (_r(r"\bextinguishment\b"),       _r(r"native title|aboriginal|land rights")),
            (_r(r"traditional laws|traditional customs"), _r(r"native title|aboriginal")),
            (_r(r"torres strait"),            _r(r"native title|aboriginal|indigenous|land")),
            (_r(r"\bmabo\b"),                 _r(r"native title|aboriginal|land")),
        ],
    ),

    "Employment & Industrial Law": dict(
        primary=[
            _r(r"industrial law"),
            _r(r"workplace relations"),
            _r(r"industrial relations"),
            _r(r"workers.? compensation"),
            _r(r"employer and employee"),
        ],
        secondary=[
            (_r(r"\bemployment\b"),           _r(r"industrial|workplace|employee|employer|dismiss|reinstat|fair work")),
            (_r(r"\breinstatement\b"),        _r(r"employ|dismiss|industrial|workplace")),
            (_r(r"unfair dismissal"),         _r(r"employ|industrial|workplace")),
            (_r(r"enterprise agreement"),     _r(r"employ|industrial|workplace|bargaining|fair work")),
            (_r(r"trade union"),              _r(r"industrial|employ|workplace")),
            (_r(r"statutory duty"),           _r(r"employer|employee|workman|workplace|occupational")),
            (_r(r"general protections"),      _r(r"fair work|employ|industrial")),
            (_r(r"adverse action"),           _r(r"fair work|employ|industrial|general protections")),
            (_r(r"independent contractor"),   _r(r"employ|worker|vicarious")),
            (_r(r"military rehabilitation|safety.*rehabilitation.*compensation"), _r(r"workers|compensation|injury|employ")),
        ],
    ),

    "Discrimination Law": dict(
        primary=[
            _r(r"discrimination law"),
            _r(r"disability discrimination"),
            _r(r"racial discrimination"),
            _r(r"sex discrimination"),
        ],
        secondary=[
            (_r(r"\bdiscrimination\b"),       _r(r"disability|racial|sex|age|religion|equal opportunity|human rights")),
            (_r(r"equal opportunity"),        _r(r"discrimination|disability|sex|race")),
            (_r(r"human rights"),             _r(r"discrimination|equal|freedom")),
        ],
    ),

    "Aviation, Shipping & Transport": dict(
        primary=[
            _r(r"\baviation\b"),
            _r(r"\bshipping\b"),
            _r(r"sea carriage"),
            _r(r"carriage of goods"),
        ],
        secondary=[
            (_r(r"bill of lading"),           _r(r"ship|cargo|carriage|maritime")),
            (_r(r"\bmaritime\b"),             _r(r"ship|cargo|carriage|sea")),
            (_r(r"civil aviation"),           _r(r"aircraft|flight|carrier")),
            (_r(r"\baircraft\b"),             _r(r"aviation|carrier|flight|civil aviation act")),
            (_r(r"\bcustoms\b"),              _r(r"importation|prohibited|handgun|weapon|goods")),
        ],
    ),

    "Resources, Planning & Local Government": dict(
        primary=[
            _r(r"\bmining\b"),
            _r(r"town planning"),
            _r(r"local government"),
            _r(r"environmental planning"),
            _r(r"development consent"),
        ],
        secondary=[
            (_r(r"\bmineral"),                _r(r"mining|crown|royalty|resource|compensation")),
            (_r(r"crown prerogative"),        _r(r"mineral|mining|resource")),
            (_r(r"\bzoning\b"),              _r(r"planning|local government|land use")),
            (_r(r"planning instrument"),      _r(r"planning|local government|zoning|development")),
            (_r(r"council resolution"),       _r(r"local government|council|planning|validity")),
            (_r(r"access regime|declaration"), _r(r"national competition|trade practices|infrastructure|resource")),
            (_r(r"mine subsidence"),          _r(r"mining|compensation|land")),
        ],
    ),

    "Practice & Procedure": dict(
        primary=[
            _r(r"practice and procedure"),
            _r(r"\bpleading\b"),
            _r(r"limitation of actions"),
            _r(r"abuse of process"),
            _r(r"contempt of court"),
            _r(r"courts and judges"),
            _r(r"private international law"),
        ],
        secondary=[
            (_r(r"\bcosts\b"),                _r(r"practice|procedure|court|appeal|order|award")),
            (_r(r"res judicata|issue estoppel"), _r(r"practice|procedure|court|judgment")),
            (_r(r"\bdiscovery\b"),            _r(r"practice|procedure|court|documents")),
            (_r(r"interlocutory"),            _r(r"practice|procedure|injunction|relief|court")),
            (_r(r"special leave"),            _r(r"high court|appeal|practice|procedure")),
            (_r(r"representative proceedings"), _r(r"practice|procedure|court|federal court")),
            (_r(r"summary judgment"),         _r(r"practice|procedure|court")),
            (_r(r"extension of time"),        _r(r"practice|procedure|appeal|court")),
            (_r(r"courts?.* appeal"),         _r(r"jurisdiction|powers|practice|procedure")),
            (_r(r"forum non conveniens"),     _r(r"jurisdiction|court|private international|tort")),
            (_r(r"appellate jurisdiction"),   _r(r"court|appeal|procedure|high court")),
            (_r(r"amicus curiae|intervener"), _r(r"court|appeal|procedure|high court")),
        ],
    ),
}


# ── Tagging function ──────────────────────────────────────────────────────────

def tag_case(catchwords: str) -> list[str]:
    """Return matching law area labels for a catchwords string."""
    if not catchwords:
        return []

    lead = leading_topic(catchwords)
    matched = []

    for area, rules in LAW_AREAS.items():
        if any(p.search(lead) for p in rules["primary"]):
            matched.append(area)
            continue
        if any(
            pa.search(catchwords) and pb.search(catchwords)
            for pa, pb in rules["secondary"]
        ):
            matched.append(area)

    return matched


# ── Apply to all cases ────────────────────────────────────────────────────────
for citation, case in cases_all.items():
    case["law_areas"] = tag_case(case.get("catchwords", "") or "")

# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = defaultdict(int)
untagged = []

for citation, case in cases_all.items():
    tags = case["law_areas"]
    if not tags:
        untagged.append(citation)
    for tag in tags:
        tag_counts[tag] += 1

print("Cases per law area:")
for area, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4}  {area}")

print(f"\nTotal cases  : {len(cases_all)}")
print(f"Tagged       : {len(cases_all) - len(untagged)}")
print(f"Untagged     : {len(untagged)}")
if untagged:
    print(f"\nUntagged (empty or unparseable catchwords):")
    for c in sorted(untagged):
        cw = cases_all[c].get('catchwords','') or ''
        print(f"  {c}: {cw[:100] if cw else '(empty)'}")"""


Cases per law area:
   286  Statutory Interpretation
   188  Criminal Law
   176  Constitutional Law
   131  Practice & Procedure
   107  Evidence
   106  Contract & Commercial Law
   102  Torts
    87  Property & Equity
    70  Taxation
    68  Immigration & Refugees
    55  Corporations & Insolvency
    43  Administrative Law
    37  Employment & Industrial Law
    21  Aviation, Shipping & Transport
    21  Intellectual Property
    15  Resources, Planning & Local Government
    15  Family Law
    14  Discrimination Law
    12  Native Title & Aboriginal Law

Total cases  : 781
Tagged       : 777
Untagged     : 4

Untagged (empty or unparseable catchwords):
  [1999] HCA 16: Thompson v His Honour Judge Byrne & Ors Criminal law – Motor traffic offence – Prescribed concentrat
  [2003] HCA 53: Russo v Aiello Limitation of actions – Action in respect of motor vehicle accident where claimant lo
  [2008] HCA 45: BHP Billiton Iron Ore Pty Ltd v National Competition Council BHP Billiton Iron O

In [12]:
cases_all["[1999] HCA 50"]

{'citation': '[1999] HCA 50',
 'catchwords': 'Katsuno v The Queen Criminal law – Juries – Practice in Victoria of provision of conviction and other information concerning potential jurors to Director of Public Prosecutions by police – Information used to exercise challenge – Whether practice prohibited by Juries Act 1967 (Vic) – Consequences of invalidation of practice – Whether prohibition of practice relates to the constitution and authority of the jury and to the trial process in a fundamental respect – Whether breach constituted a fundamental failure to observe requirements of the criminal process. Criminal law – Juries – Nature of and entitlement to peremptory challenge – Relevance of reasons for exercise. Criminal law – Constitutional requirement of trial by jury – Representative nature of the jury. Jury – Commonwealth offence – Trial in State court – Provision of information by police to prosecutor – Information used to exercise challenge – Whether practice prohibited – Whether 

In [18]:
import re
from collections import defaultdict

# ── Category keyword map ──────────────────────────────────────────────────────
# Matched case-insensitively against the FIRST topic segment of catchwords only.
# Longer/more specific phrases are matched before shorter ones automatically.
# ─────────────────────────────────────────────────────────────────────────────

CATEGORY_KEYWORDS = {

    "Constitutional Law": [
        "constitutional law",
        "legislative power", "judicial power of the commonwealth",
        "judicial power of commonwealth", "inconsistency between commonwealth",
        "chapter iii", "ch iii", "separation of powers",
        "separation of judicial power", "compulsory acquisition",
        "acquisition of property", "manner and form",
        "exclusive powers of commonwealth parliament",
        "exclusive powers of the commonwealth parliament",
        "powers of the parliament", "powers of commonwealth parliament",
        "powers of federal parliament", "parliamentary elections",
        "law of the commonwealth", "validity of law",
        "restrictions on commonwealth and state legislation",
        "operation and effect of constitution",
        "nature and extent of power",
        "state parliament", "legislative council",
        "federal jurisdiction", "federal court", "federal courts",
        "high court and federal judiciary", "high court",
        "inconsistency", "vesting of federal jurisdiction",
        "amending act", "civil trial by jury",
    ],

    "Administrative Law": [
        "administrative law",
        "judicial review", "jurisdictional error", "procedural fairness",
        "natural justice", "prerogative writ", "privative clause",
        "freedom of information", "courts and judges",
        "bias", "apprehended bias", "actual bias",
        "standing", "ultra vires", "delegated legislation",
        "certiorari", "mandamus", "prohibition",
        "review", "application for review",
        "request for surrender", "surrender determination",
        "original jurisdiction",
    ],

    "Criminal Law": [
        "criminal law", "criminal procedure", "criminal law and procedure",
        "sentencing", "sentence",
        "murder", "manslaughter", "sexual offence", "rape",
        "double jeopardy", "military law",
        "appeal against conviction", "appeal against sentence",
        "extradition", "criminal appeals", "criminal trial",
        "conspiracy to defraud", "conspiracy",
        "attempt to possess", "supply of dangerous drug",
        "supply of prohibited drug", "attempted possession",
        "dishonestly obtaining", "property offence",
        "homicide", "burglary", "perjury",
        "criminal responsibility", "physical element of offence",
        "direction to jury", "directions to jury",
        "trial by jury", "lies", "proviso",
        "motor traffic offence", "offences against state law",
        "indictable offence", "committal proceedings",
        "criminal proceedings",
        "break and enter", "unlawful killing",
        "terrorism", "crimes and offences by children",
        "joint criminal enterprise", "common intention to prosecute unlawful purpose",
        "defences",
        "conviction",
    ],

    "Immigration & Refugees": [
        "immigration", "refugees", "refugee",
        "protection visa", "refugee review tribunal",
        "citizenship and migration", "unlawful non-citizen",
        "application for protection visa", "cancellation of visas",
        "visa", "migration", "deportation", "naturalisation",
        "naturalization and aliens", "naturalisation and aliens",
        "citizenship",
    ],

    "Torts": [
        "negligence", "torts", "tort", "defamation",
        "duty of care", "malicious prosecution", "medical negligence",
        "breach of duty", "causation", "contributory negligence",
        "vicarious liability", "personal injury", "personal injuries",
        "nervous shock", "psychiatric injury",
        "nuisance", "trespass", "trespass to land",
        "misfeasance", "standard of care",
        "wrongful death", "qualified privilege",
        "res ipsa loquitur", "occupier", "occupier's liability",
        "negligent misstatement", "pure economic loss",
        "joint tortfeasors", "elements of the tort",
        "motor accident", "motor vehicles",
        "damages",
        "measure of damages in actions for tort",
        "foreign tort", "deceit", "inducement of breach of contract",
        "misrepresentation", "statements amounting to defamation",
        "defence of qualified privilege",
    ],

    "Contract & Commercial Law": [
        "contract", "contracts", "trade practices",
        "misleading or deceptive conduct", "sale of goods",
        "banker and customer", "unjust enrichment", "restitution",
        "restrictive trade practices",
        "construction of policy", "construction of lease",
        "construction and interpretation",
        "intention to create contractual relations",
        "exclusion of liability", "exclusionary provisions",
        "restraint of trade", "penalty doctrine", "penalty",
        "unconscionable conduct",
        "insurance", "professional indemnity insurance", "professional indemnity",
        "indemnity", "guarantee",
        "repudiation", "termination for breach",
        "market definition", "market power", "competition",
        "consumer protection", "fair trading act",
        "illegality",
        "credit facility", "loan",
        "precontractual disclosure",
        "access to services",
        "surety",
    ],

    "Statutory Interpretation": [
        "statutory interpretation", "statutory construction",
        "acts of parliament", "statutes",
        "construction", "interpretation",
        "construction of legislation",
        "words and phrases",
        "remedial legislation",
    ],

    "Evidence": [
        "evidence",
        "admissibility", "legal professional privilege",
        "tendency evidence", "propensity evidence",
        "hearsay", "hearsay rule", "credibility", "corroboration",
        "expert evidence", "opinion evidence",
        "burden of proof", "standard of proof", "onus of proof",
        "onus and standard of proof",
        "privilege",
        "without prejudice", "spousal privilege",
        "identification evidence",
        "prior inconsistent",
        "false evidence",
        "competence and compellability",
    ],

    "Property & Equity": [
        "real property", "torrens", "equity", "trusts", "trust",
        "land tax", "succession", "mortgages", "mortgage",
        "vesting of property", "conveyance", "conveyancing",
        "easements", "easement",
        "fiduciary", "fiduciary duty", "fiduciary duties", "fiduciary obligations",
        "constructive trust", "resulting trust",
        "specific performance", "injunctions", "injunction",
        "family provision", "adequate provision",
        "equitable remedies", "equitable mortgage",
        "distinction between trust and charge",
        "express trust", "unit trusts",
        "land titles under the torrens system",
        "avoidance of settlement of property",
        "resumption of land",
        "permissive occupancy",
        "subdivision of land",
        "confidential information",
        "contribution",
        "money had and received",
        "tenure",
        "property",
    ],

    "Corporations & Insolvency": [
        "corporations", "corporation", "corporations law", "company law",
        "winding up", "winding-up", "insolvency", "bankruptcy",
        "companies", "external administration",
        "deed of company arrangement",
        "duties of directors",
        "provable debt",
        "contraventions of civil penalty",
    ],

    "Taxation": [
        "taxation", "income tax", "stamp duty", "taxes and duties",
        "allowable deductions", "assessable income", "tax avoidance",
        "fringe benefits tax", "customs", "land tax",
        "stamp duties", "dutiable transactions",
        "deductions and rebates", "capital gains tax",
        "remittance of tax", "income",
        "assessment", "avoidance of tax",
        "pay-roll tax", "duties of excise",
        "taxable supply", "goods and services tax",
        "capital gains", "capital gains and losses",
        "income of trust estate",
        "recovery of tax debts",
        "agreement for third party to discharge taxpayer",
        "conveyance on sale",
    ],

    "Family Law": [
        "family law", "family court", "child support", "child abduction",
        "custody", "guardianship", "parenting",
        "property settlement", "maintenance",
        "marriage", "divorce", "de facto",
        "children",
        "whether family court has power",
        "courts having jurisdiction in matrimonial causes",
    ],

    "Intellectual Property": [
        "intellectual property", "copyright", "patents", "patent",
        "trade marks", "trade mark", "passing off",
        "infringement", "inventive step", "obviousness", "revocation",
        "artistic works", "sound recording",
        "designs", "design",
        "phonographic performance",
    ],

    "Native Title & Aboriginal Law": [
        "native title", "aboriginal", "aboriginals", "land rights",
        "indigenous", "pastoral lease", "extinguishment",
        "traditional laws", "traditional customs", "torres strait",
        "mabo",
    ],

    "Employment & Industrial Law": [
        "industrial law", "workplace relations", "industrial relations",
        "workers compensation", "workers' compensation",
        "employer and employee",
        "employment", "reinstatement", "unfair dismissal",
        "enterprise agreement", "trade union",
        "termination of employment",
        "industrial action", "certified agreement",
        "whether award continued to apply",
        "australian industrial relations commission",
        "general protections",
        "injury",
    ],

    "Discrimination Law": [
        "discrimination law", "disability discrimination",
        "racial discrimination", "sex discrimination",
        "discrimination", "equal opportunity", "human rights",
        "slavery",
    ],

    "Aviation, Shipping & Transport": [
        "aviation", "shipping", "sea carriage", "carriage of goods",
        "carriage by air", "bill of lading", "maritime",
        "civil aviation", "aircraft",
        "liability for damage caused by aircraft",
        "marine pollution",
        "importation of handguns",
    ],

    "Resources, Planning & Local Government": [
        "mining", "town planning", "local government",
        "environmental planning", "development consent",
        "minerals", "mineral", "crown prerogative",
        "zoning", "planning instrument", "council resolution",
        "council", "local authority",
        "third party access regime",
        "statutory power under environmental planning",
        "mine subsidence", "ownership of minerals",
    ],

    "Practice & Procedure": [
        "practice and procedure", "pleading", "limitation of actions",
        "abuse of process", "contempt of court",
        "private international law",
        "costs", "res judicata", "issue estoppel",
        "discovery", "preliminary discovery",
        "interlocutory", "special leave",
        "representative proceedings", "group proceedings",
        "summary judgment", "extension of time",
        "appeal", "appeals",
        "appellate jurisdiction",
        "amicus curiae", "intervener",
        "forum non conveniens", "choice of law",
        "stay of proceedings", "permanent stay of proceedings",
        "practice",
        "search warrants",
        "foreign state immunity",
        "international agreements",
        "external affairs",
        "notice",
        "procedure",
    ],
}

# ── Compile patterns ──────────────────────────────────────────────────────────
# Sort each category longest-first so specific phrases beat short ones
_COMPILED = {
    area: sorted(
        [(kw, re.compile(r'^\s*' + re.escape(kw), re.IGNORECASE)) for kw in keywords],
        key=lambda x: -len(x[0])
    )
    for area, keywords in CATEGORY_KEYWORDS.items()
}

# Area names that sometimes appear appended to a case name in the first segment
# e.g. "Smith v Jones Criminal law – subtopic – ..."
_AREA_IN_FIRST_SEG = re.compile(
    r'\b(Criminal law|Constitutional law|Administrative law|Negligence|Torts?|'
    r'Statutes?|Contracts?|Trade practices|Evidence|Equity|Trusts?|Real property|'
    r'Corporations?|Taxation|Immigration|Refugees?|Family law|Defamation|'
    r'Patents?|Copyright|Native title|Bankruptcy|Insolvency|Sentencing|'
    r'Industrial law|Workers.{0,2}compensation|Discrimination|Aviation|Shipping|'
    r'Mining|Planning|Extradition|Practice and procedure|Damages|Stamp duty|'
    r'Customs|Land tax|Limitation of actions)\b',
    re.IGNORECASE
)


def get_leading_topic(catchwords: str) -> str:
    """
    Extract the first legal topic segment from catchwords.
    Handles 'Negligence – ...' and 'Smith v Jones Negligence – ...' forms.
    """
    if not catchwords:
        return ""
    parts = re.split(r' [–\-] ', catchwords)
    for part in parts:
        part = part.strip()
        if re.match(r'^[A-Z][a-z]', part) and ' v ' not in part and len(part) < 100:
            return part
    m = _AREA_IN_FIRST_SEG.search(parts[0])
    return m.group(0) if m else parts[0].strip()


def tag_case(catchwords: str) -> list[str]:
    """
    Tag a case by matching the leading topic phrase against category keywords.
    Returns all matching categories (usually just one).
    """
    topic = get_leading_topic(catchwords)
    if not topic:
        return []
    matched = []
    for area, patterns in _COMPILED.items():
        for kw, pattern in patterns:
            if pattern.match(topic):
                matched.append(area)
                break
    return matched


# ── Apply to all cases ────────────────────────────────────────────────────────
for citation, case in cases_all.items():
    cw = case.get("catchwords", "") or ""
    case["leading_topic"] = get_leading_topic(cw)
    case["law_areas"] = tag_case(cw)

# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = defaultdict(int)
untagged = []

for citation, case in cases_all.items():
    tags = case["law_areas"]
    if not tags:
        untagged.append(citation)
    for tag in tags:
        tag_counts[tag] += 1

print("Cases per law area:")
for area, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4}  {area}")

print(f"\nTotal cases  : {len(cases_all)}")
print(f"Tagged       : {len(cases_all) - len(untagged)}")
print(f"Untagged     : {len(untagged)}")
if untagged:
    print(f"\nUntagged (leading topic not matched):")
    for c in sorted(untagged):
        print(f"  {c}: {cases_all[c].get('leading_topic', '')}")

Cases per law area:
    92  Criminal Law
    83  Constitutional Law
    83  Torts
    57  Practice & Procedure
    47  Contract & Commercial Law
    42  Evidence
    41  Immigration & Refugees
    40  Taxation
    40  Property & Equity
    31  Administrative Law
    30  Statutory Interpretation
    19  Employment & Industrial Law
    18  Intellectual Property
    18  Corporations & Insolvency
     9  Aviation, Shipping & Transport
     8  Resources, Planning & Local Government
     8  Family Law
     8  Native Title & Aboriginal Law
     4  Discrimination Law

Total cases  : 781
Tagged       : 671
Untagged     : 110

Untagged (leading topic not matched):
  [1998] HCA 25: Application on behalf of extradition country for issue of warrant of arrest
  [1998] HCA 44: Re JJT & Ors; Ex Parte Victoria Legal Aid Costs
  [1998] HCA 73: Re East & Ors; Ex parte Nguyen Constitutional law
  [1999] HCA 10: Relief
  [1999] HCA 11: Jurisdiction
  [1999] HCA 19: District Court of New South Wales
  [1999

In [51]:
# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = defaultdict(int)
untagged = []

for citation, case in cases_all.items():
    tags = case["law_areas"]
    if not tags:
        untagged.append(citation)
    for tag in tags:
        tag_counts[tag] += 1

print("Cases per law area:")
for area, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4}  {area}")

print(f"\nTotal cases  : {len(cases_all)}")
print(f"Tagged       : {len(cases_all) - len(untagged)}")
print(f"Untagged     : {len(untagged)}")
if untagged:
    print(f"\nUntagged (leading topic not matched):")
    for c in sorted(untagged):
        print(f"  {c}: {cases_all[c].get('leading_topic', '')}")

Cases per law area:
    93  Criminal Law
    83  Constitutional Law
    83  Torts
    57  Practice & Procedure
    51  Contract & Commercial Law
    42  Taxation
    42  Evidence
    41  Immigration & Refugees
    40  Property & Equity
    31  Statutory Interpretation
    31  Administrative Law
    19  Employment & Industrial Law
    19  Corporations & Insolvency
    18  Intellectual Property
     9  Aviation, Shipping & Transport
     8  Resources, Planning & Local Government
     8  Family Law
     8  Native Title & Aboriginal Law
     4  Discrimination Law

Total cases  : 781
Tagged       : 680
Untagged     : 101

Untagged (leading topic not matched):
  [1998] HCA 25: Application on behalf of extradition country for issue of warrant of arrest
  [1998] HCA 44: Re JJT & Ors; Ex Parte Victoria Legal Aid Costs
  [1998] HCA 73: Re East & Ors; Ex parte Nguyen Constitutional law
  [1999] HCA 10: Relief
  [1999] HCA 11: Jurisdiction
  [1999] HCA 19: District Court of New South Wales
  [1999

Back up 

In [ ]:
import json

with open("cases_all_updated.json", "w", encoding="utf-8") as f:
    json.dump(cases_all, f, indent=2, ensure_ascii=False)

NameError: name 'updated_cases_all' is not defined

In [74]:
with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [ ]:
import re
from collections import defaultdict

# ── Category keyword map ──────────────────────────────────────────────────────
# Matched case-insensitively against the FIRST topic segment of catchwords only.
# Longer/more specific phrases are matched before shorter ones automatically.
# ─────────────────────────────────────────────────────────────────────────────

CATEGORY_KEYWORDS = {

    "Constitutional Law": [
        "constitutional law",
        "legislative power", "judicial power of the commonwealth",
        "judicial power of commonwealth", "inconsistency between commonwealth",
        "chapter iii", "ch iii", "separation of powers",
        "separation of judicial power", "compulsory acquisition",
        "acquisition of property", "manner and form",
        "exclusive powers of commonwealth parliament",
        "exclusive powers of the commonwealth parliament",
        "powers of the parliament", "powers of commonwealth parliament",
        "powers of federal parliament", "parliamentary elections",
        "law of the commonwealth", "validity of law",
        "restrictions on commonwealth and state legislation",
        "operation and effect of constitution",
        "nature and extent of power",
        "state parliament", "legislative council",
        "federal jurisdiction", "federal court", "federal courts",
        "high court and federal judiciary", "high court",
        "inconsistency", "vesting of federal jurisdiction",
        "amending act", "civil trial by jury",
    ],

    "Administrative Law": [
        "administrative law",
        "judicial review", "jurisdictional error", "procedural fairness",
        "natural justice", "prerogative writ", "privative clause",
        "freedom of information", "courts and judges",
        "bias", "apprehended bias", "actual bias",
        "standing", "ultra vires", "delegated legislation",
        "certiorari", "mandamus", "prohibition",
        "review", "application for review",
        "request for surrender", "surrender determination",
        "original jurisdiction",
    ],

    "Criminal Law": [
        "criminal law", "criminal procedure", "criminal law and procedure",
        "sentencing", "sentence",
        "murder", "manslaughter", "sexual offence", "rape",
        "double jeopardy", "military law",
        "appeal against conviction", "appeal against sentence",
        "extradition", "criminal appeals", "criminal trial",
        "conspiracy to defraud", "conspiracy",
        "attempt to possess", "supply of dangerous drug",
        "supply of prohibited drug", "attempted possession",
        "dishonestly obtaining", "property offence",
        "homicide", "burglary", "perjury",
        "criminal responsibility", "physical element of offence",
        "direction to jury", "directions to jury",
        "trial by jury", "lies", "proviso",
        "motor traffic offence", "offences against state law",
        "indictable offence", "committal proceedings",
        "criminal proceedings",
        "break and enter", "unlawful killing",
        "terrorism", "crimes and offences by children",
        "joint criminal enterprise", "common intention to prosecute unlawful purpose",
        "defences",
        "conviction",
    ],

    "Immigration & Refugees": [
        "immigration", "refugees", "refugee",
        "protection visa", "refugee review tribunal",
        "citizenship and migration", "unlawful non-citizen",
        "application for protection visa", "cancellation of visas",
        "visa", "migration", "deportation", "naturalisation",
        "naturalization and aliens", "naturalisation and aliens",
        "citizenship",
    ],

    "Torts": [
        "negligence", "torts", "tort", "defamation",
        "duty of care", "malicious prosecution", "medical negligence",
        "breach of duty", "causation", "contributory negligence",
        "vicarious liability", "personal injury", "personal injuries",
        "nervous shock", "psychiatric injury",
        "nuisance", "trespass", "trespass to land",
        "misfeasance", "standard of care",
        "wrongful death", "qualified privilege",
        "res ipsa loquitur", "occupier", "occupier's liability",
        "negligent misstatement", "pure economic loss",
        "joint tortfeasors", "elements of the tort",
        "motor accident", "motor vehicles",
        "damages",
        "measure of damages in actions for tort",
        "foreign tort", "deceit", "inducement of breach of contract",
        "misrepresentation", "statements amounting to defamation",
        "defence of qualified privilege",
    ],

    "Contract & Commercial Law": [
        "contract", "contracts", "trade practices",
        "misleading or deceptive conduct", "sale of goods",
        "banker and customer", "unjust enrichment", "restitution",
        "restrictive trade practices",
        "construction of policy", "construction of lease",
        "construction and interpretation",
        "intention to create contractual relations",
        "exclusion of liability", "exclusionary provisions",
        "restraint of trade", "penalty doctrine", "penalty",
        "unconscionable conduct",
        "insurance", "professional indemnity insurance", "professional indemnity",
        "indemnity", "guarantee",
        "repudiation", "termination for breach",
        "market definition", "market power", "competition",
        "consumer protection", "fair trading act",
        "illegality",
        "credit facility", "loan",
        "precontractual disclosure",
        "access to services",
        "surety",
    ],

    "Statutory Interpretation": [
        "statutory interpretation", "statutory construction",
        "acts of parliament", "statutes",
        "construction", "interpretation",
        "construction of legislation",
        "words and phrases",
        "remedial legislation",
    ],

    "Evidence": [
        "evidence",
        "admissibility", "legal professional privilege",
        "tendency evidence", "propensity evidence",
        "hearsay", "hearsay rule", "credibility", "corroboration",
        "expert evidence", "opinion evidence",
        "burden of proof", "standard of proof", "onus of proof",
        "onus and standard of proof",
        "privilege",
        "without prejudice", "spousal privilege",
        "identification evidence",
        "prior inconsistent",
        "false evidence",
        "competence and compellability",
    ],

    "Property & Equity": [
        "real property", "torrens", "equity", "trusts", "trust",
        "land tax", "succession", "mortgages", "mortgage",
        "vesting of property", "conveyance", "conveyancing",
        "easements", "easement",
        "fiduciary", "fiduciary duty", "fiduciary duties", "fiduciary obligations",
        "constructive trust", "resulting trust",
        "specific performance", "injunctions", "injunction",
        "family provision", "adequate provision",
        "equitable remedies", "equitable mortgage",
        "distinction between trust and charge",
        "express trust", "unit trusts",
        "land titles under the torrens system",
        "avoidance of settlement of property",
        "resumption of land",
        "permissive occupancy",
        "subdivision of land",
        "confidential information",
        "contribution",
        "money had and received",
        "tenure",
        "property",
    ],

    "Corporations & Insolvency": [
        "corporations", "corporation", "corporations law", "company law",
        "winding up", "winding-up", "insolvency", "bankruptcy",
        "companies", "external administration",
        "deed of company arrangement",
        "duties of directors",
        "provable debt",
        "contraventions of civil penalty",
    ],

    "Taxation": [
        "taxation", "income tax", "stamp duty", "taxes and duties",
        "allowable deductions", "assessable income", "tax avoidance",
        "fringe benefits tax", "customs", "land tax",
        "stamp duties", "dutiable transactions",
        "deductions and rebates", "capital gains tax",
        "remittance of tax", "income",
        "assessment", "avoidance of tax",
        "pay-roll tax", "duties of excise",
        "taxable supply", "goods and services tax",
        "capital gains", "capital gains and losses",
        "income of trust estate",
        "recovery of tax debts",
        "agreement for third party to discharge taxpayer",
        "conveyance on sale",
    ],

    "Family Law": [
        "family law", "family court", "child support", "child abduction",
        "custody", "guardianship", "parenting",
        "property settlement", "maintenance",
        "marriage", "divorce", "de facto",
        "children",
        "whether family court has power",
        "courts having jurisdiction in matrimonial causes",
    ],

    "Intellectual Property": [
        "intellectual property", "copyright", "patents", "patent",
        "trade marks", "trade mark", "passing off",
        "infringement", "inventive step", "obviousness", "revocation",
        "artistic works", "sound recording",
        "designs", "design",
        "phonographic performance",
    ],

    "Native Title & Aboriginal Law": [
        "native title", "aboriginal", "aboriginals", "land rights",
        "indigenous", "pastoral lease", "extinguishment",
        "traditional laws", "traditional customs", "torres strait",
        "mabo",
    ],

    "Employment & Industrial Law": [
        "industrial law", "workplace relations", "industrial relations",
        "workers compensation", "workers' compensation",
        "employer and employee",
        "employment", "reinstatement", "unfair dismissal",
        "enterprise agreement", "trade union",
        "termination of employment",
        "industrial action", "certified agreement",
        "whether award continued to apply",
        "australian industrial relations commission",
        "general protections",
        "injury",
    ],

    "Discrimination Law": [
        "discrimination law", "disability discrimination",
        "racial discrimination", "sex discrimination",
        "discrimination", "equal opportunity", "human rights",
        "slavery",
    ],

    "Aviation, Shipping & Transport": [
        "aviation", "shipping", "sea carriage", "carriage of goods",
        "carriage by air", "bill of lading", "maritime",
        "civil aviation", "aircraft",
        "liability for damage caused by aircraft",
        "marine pollution",
        "importation of handguns",
    ],

    "Resources, Planning & Local Government": [
        "mining", "town planning", "local government",
        "environmental planning", "development consent",
        "minerals", "mineral", "crown prerogative",
        "zoning", "planning instrument", "council resolution",
        "council", "local authority",
        "third party access regime",
        "statutory power under environmental planning",
        "mine subsidence", "ownership of minerals",
    ],

    "Practice & Procedure": [
        "practice and procedure", "pleading", "limitation of actions",
        "abuse of process", "contempt of court",
        "private international law",
        "costs", "res judicata", "issue estoppel",
        "discovery", "preliminary discovery",
        "interlocutory", "special leave",
        "representative proceedings", "group proceedings",
        "summary judgment", "extension of time",
        "appeal", "appeals",
        "appellate jurisdiction",
        "amicus curiae", "intervener",
        "forum non conveniens", "choice of law",
        "stay of proceedings", "permanent stay of proceedings",
        "practice",
        "search warrants",
        "foreign state immunity",
        "international agreements",
        "external affairs",
        "notice",
        "procedure",
    ],
}

# ── Compile patterns ──────────────────────────────────────────────────────────
# Sort each category longest-first so specific phrases beat short ones
_COMPILED = {
    area: sorted(
        [(kw, re.compile(r'^\s*' + re.escape(kw), re.IGNORECASE)) for kw in keywords],
        key=lambda x: -len(x[0])
    )
    for area, keywords in CATEGORY_KEYWORDS.items()
}

# Area names that sometimes appear appended to a case name in the first segment
# e.g. "Smith v Jones Criminal law – subtopic – ..."
_AREA_IN_FIRST_SEG = re.compile(
    r'\b(Criminal law|Constitutional law|Administrative law|Negligence|Torts?|'
    r'Statutes?|Contracts?|Trade practices|Evidence|Equity|Trusts?|Real property|'
    r'Corporations?|Taxation|Immigration|Refugees?|Family law|Defamation|'
    r'Patents?|Copyright|Native title|Bankruptcy|Insolvency|Sentencing|'
    r'Industrial law|Workers.{0,2}compensation|Discrimination|Aviation|Shipping|'
    r'Mining|Planning|Extradition|Practice and procedure|Damages|Stamp duty|'
    r'Customs|Land tax|Limitation of actions)\b',
    re.IGNORECASE
)


def get_leading_topic(catchwords: str) -> str:
    """
    Extract the first legal topic segment from catchwords.
    Handles 'Negligence – ...' and 'Smith v Jones Negligence – ...' forms.
    """
    if not catchwords:
        return ""
    parts = re.split(r' [–\-] ', catchwords)
    for part in parts:
        part = part.strip()
        if re.match(r'^[A-Z][a-z]', part) and ' v ' not in part and len(part) < 100:
            return part
    m = _AREA_IN_FIRST_SEG.search(parts[0])
    return m.group(0) if m else parts[0].strip()

_TITLE_RULES = [
    (re.compile(r'\bconstitutional\b', re.IGNORECASE), "Constitutional Law"),
    (re.compile(r'\btaxation\b|\btax\b', re.IGNORECASE), "Taxation"),
    (re.compile(r'\bcriminal\b',       re.IGNORECASE), "Criminal Law"),
    (re.compile(r'\bextradition\b',    re.IGNORECASE), "Extradition"),
    (re.compile(r'\bprocedure\b',    re.IGNORECASE), "Practice & Procedure"),
    (re.compile(r'\bemployment\b',    re.IGNORECASE), "Employment & Industrial Law"),
    (re.compile(r'\bpatents?\b|\bcopyright\b',    re.IGNORECASE), "Intellectual Property"),
    (re.compile(r'\bnegligence\b|\btorts?\b',    re.IGNORECASE), "Torts"),
    (re.compile(r'\badministrative law\b|\bjudicial review\b', re.IGNORECASE), "Administrative Law"),
    (re.compile(r'\binsurance\b', re.IGNORECASE), "Insurance Law"),
    (re.compile(r'\bcontract\b|\btrade practices\b', re.IGNORECASE), "Contract & Commercial Law")
]

# Fallback: area name phrases to search anywhere in catchwords for untagged cases
_FULLTEXT_FALLBACK = [
    (re.compile(r'\bcriminal law\b',                   re.IGNORECASE), "Criminal Law"),
    (re.compile(r'\bconstitutional law\b',             re.IGNORECASE), "Constitutional Law"),
    (re.compile(r'\bnegligence\b|\btorts?\b|\bduty of care\b|\bdefamation\b', re.IGNORECASE), "Torts"),
    (re.compile(r'\bpractice and procedure\b',         re.IGNORECASE), "Practice & Procedure"),
    (re.compile(r'\bcontract\b|\btrade practices\b', re.IGNORECASE), "Contract & Commercial Law"),
    (re.compile(r'\bevidence\b',                       re.IGNORECASE), "Evidence"),
    (re.compile(r'\bimmigration\b|\brefugee\b',      re.IGNORECASE), "Immigration & Refugees"),
    (re.compile(r'\btaxation\b|\bincome tax\b',      re.IGNORECASE), "Taxation"),
    (re.compile(r'\breal property\b|\bequity\b|\btrusts?\b', re.IGNORECASE), "Property & Equity"),
    (re.compile(r'\badministrative law\b|\bjudicial review\b', re.IGNORECASE), "Administrative Law"),
    (re.compile(r'\bstatutory (interpretation|construction)\b',  re.IGNORECASE), "Statutory Interpretation"),
    (re.compile(r'\bindustrial law\b|\bworkers.{0,2}compensation\b|\bemployer and employee\b', re.IGNORECASE), "Employment & Industrial Law"),
    (re.compile(r'\bcopyright\b|\bpatents?\b|\btrade marks?\b', re.IGNORECASE), "Intellectual Property"),
    (re.compile(r'\bcorporations? law\b|\bwinding up\b|\binsolvency\b|\bbankruptcy\b', re.IGNORECASE), "Corporations & Insolvency"),
    (re.compile(r'\baviation\b|\bshipping\b|\bsea carriage\b', re.IGNORECASE), "Aviation, Shipping & Transport"),
    (re.compile(r'\bmining\b|\btown planning\b|\blocal government\b', re.IGNORECASE), "Resources, Planning & Local Government"),
    (re.compile(r'\bfamily law\b|\bfamily court\b',  re.IGNORECASE), "Family Law"),
    (re.compile(r'\bnative title\b|\baboriginal\b',  re.IGNORECASE), "Native Title & Aboriginal Law"),
    (re.compile(r'\bdiscrimination law\b',                 re.IGNORECASE), "Discrimination Law"),
    (re.compile(r'\bextradition\b',    re.IGNORECASE), "Extradition"),
    (re.compile(r'\binsurance\b',                 re.IGNORECASE), "Insurance Law")
]


def tag_case(catchwords: str, title: str = "") -> list[str]:
    """
    Tag a case by matching the leading topic phrase against category keywords.
    Also applies title-based rules (constitutional, tax/duty, criminal, extradition).
    Returns all matching categories (usually just one).
    """
    matched = []
 
    # Title-based rules
    for pattern, area in _TITLE_RULES:
        if title and pattern.search(title) and area not in matched:
            matched.append(area)
 
    topic = get_leading_topic(catchwords)
    if topic:
        for area, patterns in _COMPILED.items():
            if area in matched:
                continue  # already tagged, don't duplicate
            for kw, pattern in patterns:
                if pattern.match(topic):
                    matched.append(area)
                    break
 
    # Fallback: if still untagged, scan full catchwords for area name phrases
    if not matched and catchwords:
        for pattern, area in _FULLTEXT_FALLBACK:
            if pattern.search(catchwords) and area not in matched:
                matched.append(area)
 
    return matched


# ── Apply to all cases ────────────────────────────────────────────────────────
for citation, case in backup_cases_all.items():
    cw = case.get("catchwords", "") or ""
    title = (cw.split(" \u2013 ")[0] if " \u2013 " in cw else cw.split(" - ")[0]).strip()
    case["leading_topic"] = get_leading_topic(cw)
    existing = case.get("law_areas", [])
    new_tags = tag_case(cw, title=title)
    case["law_areas"] = existing + [t for t in new_tags if t not in existing]

# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = defaultdict(int)
untagged = []

for citation, case in backup_cases_all.items():
    tags = case["law_areas"]
    if not tags:
        untagged.append(citation)
    for tag in tags:
        tag_counts[tag] += 1

print("Cases per law area:")
for area, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4}  {area}")

print(f"\nTotal cases  : {len(backup_cases_all)}")
print(f"Tagged       : {len(backup_cases_all) - len(untagged)}")
print(f"Untagged     : {len(untagged)}")
if untagged:
    print(f"\nUntagged (leading topic not matched):")
    for c in sorted(untagged):
        print(f"  {c}: {backup_cases_all[c].get('leading_topic', '')}")


Cases per law area:
   162  Criminal Law
   127  Constitutional Law
    94  Torts
    83  Practice & Procedure
    71  Contract & Commercial Law
    60  Taxation
    49  Evidence
    49  Immigration & Refugees
    48  Property & Equity
    42  Administrative Law
    35  Statutory Interpretation
    26  Employment & Industrial Law
    23  Insurance Law
    22  Intellectual Property
    21  Corporations & Insolvency
    10  Resources, Planning & Local Government
     9  Family Law
     9  Aviation, Shipping & Transport
     8  Native Title & Aboriginal Law
     8  Other
     4  Discrimination Law
     1  Resources, Planning & Local
     1  constitutional law
     1  taxation

Total cases  : 781
Tagged       : 775
Untagged     : 6


In [239]:
# ── Summary ───────────────────────────────────────────────────────────────────
tag_counts = defaultdict(int)
untagged = []

for citation, case in backup_cases_all.items():
    tags = case["law_areas"]
    if not tags:
        untagged.append(citation)
    for tag in tags:
        tag_counts[tag] += 1

print("Cases per law area:")
for area, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
    print(f"  {count:4}  {area}")

print(f"\nTotal cases  : {len(backup_cases_all)}")
print(f"Tagged       : {len(backup_cases_all) - len(untagged)}")
print(f"Untagged     : {len(untagged)}")
if untagged:
    print(f"\nUntagged (leading topic not matched):")
    for c in sorted(untagged):
        print(f"  {c}: {backup_cases_all[c].get('leading_topic', '')}")

Cases per law area:
   162  Criminal Law
   128  Constitutional Law
    95  Torts
    86  Practice & Procedure
    71  Contract & Commercial Law
    62  Taxation
    49  Evidence
    49  Immigration & Refugees
    48  Property & Equity
    42  Administrative Law
    36  Statutory Interpretation
    26  Employment & Industrial Law
    23  Insurance Law
    22  Intellectual Property
    21  Corporations & Insolvency
    11  Resources, Planning & Local Government
     9  Family Law
     9  Aviation, Shipping & Transport
     8  Native Title & Aboriginal Law
     8  Extradition
     4  Discrimination Law

Total cases  : 781
Tagged       : 781
Untagged     : 0


In [232]:
case = "[1999] HCA 19"
print(backup_cases_all[case].get("catchwords", ""))
print(backup_cases_all[case].get("leading_topic", ""))
print(backup_cases_all[case]["law_areas"])
backup_cases_all[case]["law_areas"] = ["Practice & Procedure"]
print(backup_cases_all[case]["law_areas"])

Pelechowski v The Registrar, Court of Appeal Courts and tribunals – District Court of New South Wales – Powers – Asset preservation order made after judgment – Whether order beyond power. Courts and tribunals – Powers – Implied power – Asset preservation order made after judgment – Whether power necessary for the effective exercise of jurisdiction – Relevance of express powers. Contempt – District Court of New South Wales – Breach of order – Order beyond power – Whether contempt committed – Whether order a nullity or effective until set aside. Costs – Availability – Proceedings criminal in nature but not a criminal prosecution. Words and phrases – "necessary". District Court Act 1973 (NSW), ss 46, 203. Interpretation Act 1987 (NSW), ss 34, 35.
District Court of New South Wales
['Statutory Interpretation']
['Practice & Procedure']


In [238]:
# renaming misnamed areas to match the category names
RENAME = {
    'constitutional law': 'Constitutional Law',
    'taxation': 'Taxation',
    'Resources, Planning & Local': 'Resources, Planning & Local Government',
    'Other': 'Extradition',
}

for case in backup_cases_all.values():
    case['law_areas'] = [RENAME.get(tag, tag) for tag in case['law_areas']]

In [243]:
import json

with open("cases_all_updated.json", "w", encoding="utf-8") as f:
    json.dump(backup_cases_all, f, indent=2, ensure_ascii=False)

with open("cases_all_updated.json", encoding="utf-8") as f:
        backup_cases_all = json.load(f)
backup_cases_all.keys()

dict_keys(['[1998] HCA 28', '[2003] HCA 2', '[2003] HCA 22', '[2001] HCA 30', '[2009] HCA 27', '[2005] HCA 25', '[2010] HCA 16', '[1998] HCA 11', '[2000] HCA 63', '[2006] HCA 63', '[1999] HCA 14', '[2001] HCA 17', '[2000] HCA 54', '[1998] HCA 57', '[2010] HCA 45', '[2004] HCA 52', '[2009] HCA 41', '[2001] HCA 64', '[1999] HCA 21', '[2000] HCA 57', '[2011] HCA 39', '[2000] HCA 47', '[2010] HCA 28', '[2007] HCA 22', '[2010] HCA 1', '[2005] HCA 24', '[2003] HCA 6', '[2000] HCA 48', '[2010] HCA 4', '[1998] HCA 67', '[1999] HCA 54', '[2000] HCA 40', '[2008] HCA 31', '[1999] HCA 29', '[2011] HCA 49', '[2011] HCA 4', '[2002] HCA 11', '[2001] HCA 63', '[2004] HCA 35', '[2002] HCA 53', '[2006] HCA 27', '[2000] HCA 41', '[2002] HCA 36', '[1999] HCA 27', '[1998] HCA 27', '[1999] HCA 26', '[1999] HCA 66', '[2001] HCA 22', '[1999] HCA 67', '[2002] HCA 35', '[2011] HCA 13', '[2010] HCA 41', '[2011] HCA 48', '[2006] HCA 46', '[1998] HCA 55', '[1999] HCA 9', '[2011] HCA 21', '[1998] HCA 30', '[2007] H

In [242]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
from itertools import combinations

# --- Load ---
with open('cases_all_updated.json') as f:
    data = json.load(f)

all_tag_lists = [case['law_areas'] for case in data.values()]
all_tags_flat = [t for tags in all_tag_lists for t in tags]
tag_freq = Counter(all_tags_flat)
unique_tags = sorted(tag_freq, key=lambda t: -tag_freq[t])
tag_counts_per_case = [len(tags) for tags in all_tag_lists]

# ============================================================
# FIGURE 1: Tags per case + frequency bar chart
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#F7F9FC')

# Left: distribution of tags per case
ax = axes[0]
ax.set_facecolor('#F7F9FC')
dist = Counter(tag_counts_per_case)
x_vals = sorted(dist)
y_vals = [dist[x] for x in x_vals]
bars = ax.bar(x_vals, y_vals,
              color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'],
              edgecolor='white', linewidth=1.2, zorder=3)
for bar, val in zip(bars, y_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 4,
            f'{val}\n({100 * val / len(data):.1f}%)',
            ha='center', va='bottom', fontsize=10, color='#333')
ax.set_xlabel('Number of Tags per Case', fontsize=12)
ax.set_ylabel('Number of Cases', fontsize=12)
ax.set_title('Tags per Case — Distribution', fontsize=14, fontweight='bold', pad=12)
ax.set_xticks(x_vals)
ax.set_xticklabels([f'{x} tag{"s" if x > 1 else ""}' for x in x_vals])
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
stats_text = (
    f'Total cases: {len(data)}\n'
    f'Mean tags/case: {np.mean(tag_counts_per_case):.2f}\n'
    f'Median: {np.median(tag_counts_per_case):.1f}\n'
    f'Max: {max(tag_counts_per_case)}'
)
ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
        ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#ccc'))

# Right: tag frequency
ax2 = axes[1]
ax2.set_facecolor('#F7F9FC')
colors = plt.cm.tab20.colors
freqs = [tag_freq[t] for t in unique_tags]
ax2.barh(list(reversed(unique_tags)), list(reversed(freqs)),
         color=list(reversed([colors[i % len(colors)] for i in range(len(unique_tags))])),
         edgecolor='white', linewidth=0.8, zorder=3)
for bar, val in zip(ax2.patches, list(reversed(freqs))):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', ha='left', fontsize=8.5, color='#333')
ax2.set_xlabel('Number of Cases', fontsize=12)
ax2.set_title('Tag Frequency (all cases)', fontsize=14, fontweight='bold', pad=12)
ax2.xaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax2.set_axisbelow(True)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.tick_params(axis='y', labelsize=9)

plt.tight_layout(pad=2.5)
plt.savefig('qc_tags_distribution.png', dpi=150, bbox_inches='tight', facecolor='#F7F9FC')
plt.close()
print("Saved: qc_tags_distribution.png")

# ============================================================
# FIGURE 2: Tag co-occurrence heatmap
# ============================================================
top_tags = [t for t in unique_tags if tag_freq[t] >= 5]
n = len(top_tags)
tag_index = {t: i for i, t in enumerate(top_tags)}
comat = np.zeros((n, n), dtype=int)

for tags in all_tag_lists:
    filtered = [t for t in tags if t in tag_index]
    for t1, t2 in combinations(filtered, 2):
        i, j = tag_index[t1], tag_index[t2]
        comat[i, j] += 1
        comat[j, i] += 1
    for t in filtered:
        comat[tag_index[t], tag_index[t]] += 1

off_diag = comat.copy()
np.fill_diagonal(off_diag, 0)

fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('#F7F9FC')
ax.set_facecolor('#F7F9FC')
im = ax.imshow(off_diag, cmap='YlOrRd', aspect='auto', vmin=0)

for i in range(n):
    for j in range(n):
        val = comat[i, j]
        if val > 0:
            if i == j:
                ax.add_patch(mpatches.Rectangle(
                    (i - 0.5, i - 0.5), 1, 1,
                    facecolor='#d0e8f7', edgecolor='none', zorder=1))
                ax.text(j, i, str(val), ha='center', va='center',
                        fontsize=8, color='#1a3a5c', fontweight='bold', zorder=2)
            else:
                color = 'white' if off_diag[i, j] > off_diag.max() * 0.6 else '#333'
                ax.text(j, i, str(val), ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(top_tags, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(top_tags, fontsize=9)
ax.set_title(
    'Tag Co-occurrence Matrix\n(diagonal = single tag count; off-diagonal = cases sharing both tags)',
    fontsize=13, fontweight='bold', pad=14)
plt.colorbar(im, ax=ax, shrink=0.7, label='Co-occurrence count (off-diagonal)')
plt.tight_layout(pad=2.5)
plt.savefig('qc_tag_cooccurrence.png', dpi=150, bbox_inches='tight', facecolor='#F7F9FC')
plt.close()
print("Saved: qc_tag_cooccurrence.png")

Matplotlib is building the font cache; this may take a moment.


Saved: qc_tags_distribution.png
Saved: qc_tag_cooccurrence.png
